---
description: Example cookbook for the LlamaIndex Langfuse integration using the instrumentation module of LlamaIndex.
category: Integrations
---

# Cookbook LlamaIndex Integration (Instrumentation Module)

This is a simple cookbook that demonstrates how to trace LlamaIndex applications with Langfuse based on the [instrumentation module](https://docs.llamaindex.ai/en/stable/module_guides/observability/instrumentation/) of LlamaIndex. It uses the [OpenInference LlamaIndex instrumentation](https://docs.arize.com/phoenix/tracing/integrations-tracing/llamaindex), which builds on LlamaIndex's instrumentation module and exports OpenTelemetry (OTel) spans to Langfuse. See the [LlamaIndex integration docs](https://langfuse.com/integrations/frameworks/llamaindex) for more details.

## Setup

Make sure you have `llama-index`, `langfuse`, and `openinference-instrumentation-llama-index` installed.

In [ ]:
%pip install langfuse openinference-instrumentation-llama-index llama-index --upgrade

Initialize the integration. Get your API keys from the Langfuse project settings. This example uses OpenAI for embeddings and chat completions. You can also use any other model supported by LlamaIndex.

In [2]:
import os

# Get keys for your project from the project settings page
# https://cloud.langfuse.com
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "")
os.environ.setdefault("LANGFUSE_SECRET_KEY", "")
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com") # 🇪🇺 EU region
# Other Langfuse data regions include 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com and ⚕️ HIPAA: https://hipaa.cloud.langfuse.com

# Your openai key
os.environ.setdefault("OPENAI_API_KEY", "")

Initialize the Langfuse client and register the OpenInference `LlamaIndexInstrumentor`. The instrumentor hooks into LlamaIndex's instrumentation module and exports OTel spans to Langfuse:

In [ ]:
from langfuse import get_client
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

# Initialize LlamaIndex instrumentation
LlamaIndexInstrumentor().instrument()

## Index

In [4]:
# Example context, thx ChatGPT
from llama_index.core import Document

doc1 = Document(text="""
Maxwell "Max" Silverstein, a lauded movie director, screenwriter, and producer, was born on October 25, 1978, in Boston, Massachusetts. A film enthusiast from a young age, his journey began with home movies shot on a Super 8 camera. His passion led him to the University of Southern California (USC), majoring in Film Production. Eventually, he started his career as an assistant director at Paramount Pictures. Silverstein's directorial debut, “Doors Unseen,” a psychological thriller, earned him recognition at the Sundance Film Festival and marked the beginning of a successful directing career.
""")
doc2 = Document(text="""
Throughout his career, Silverstein has been celebrated for his diverse range of filmography and unique narrative technique. He masterfully blends suspense, human emotion, and subtle humor in his storylines. Among his notable works are "Fleeting Echoes," "Halcyon Dusk," and the Academy Award-winning sci-fi epic, "Event Horizon's Brink." His contribution to cinema revolves around examining human nature, the complexity of relationships, and probing reality and perception. Off-camera, he is a dedicated philanthropist living in Los Angeles with his wife and two children.
""")

In [5]:
# Example index construction + LLM query
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents([doc1,doc2])

## Query

In [9]:
# Query
response = index.as_query_engine().query("What did he do growing up?")
print(response)

He made home movies using a Super 8 camera.


Example trace: https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/d933c7cc-20bf-4db3-810d-bab1c8d9a2a1

In [10]:
# Chat
response = index.as_chat_engine().chat("What did he do growing up?")
print(response)

He made home movies using a Super 8 camera growing up.


Example trace: https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/4e285b8f-9789-4cf0-a8b4-45473ac420f1

![LlamaIndex Chat Engine Trace in Langfuse (via instrumentation module)](https://langfuse.com/images/cookbook/integration_llama-index_instrumentation_chatengine_trace.png)

## Custom trace properties

You can set custom trace attributes such as `user_id`, `session_id`, or `tags` with the `propagate_attributes` context manager. To group multiple operations into a single trace and score it, wrap them in a root observation via `langfuse.start_as_current_observation` and use `score_trace` on the root span.

In [ ]:
from langfuse import propagate_attributes

with langfuse.start_as_current_observation(as_type="span", name="llama-index-query") as span:
    with propagate_attributes(user_id="my-user", session_id="my-session"):
        response = index.as_query_engine().query("What did he do growing up?")
    span.update(output=str(response))

    # Score the trace, e.g. based on user feedback
    span.score_trace(name="my-score", value=0.5)

langfuse.flush()

Example trace: https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/6f554d6b-a2bc-4fba-904f-aa54de2897ca